# End of week 1 exercise

To demonstrate your familiarity with your AI API choice (Z.AI), OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [2]:
# constants

MODEL_GPT = 'glm-5'
MODEL_LLAMA = 'qwen-2.5-coder'

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('ZAI_API_KEY')

# Z.AI client for GLM models
client = OpenAI(
    api_key=api_key,
    base_url="https://api.z.ai/api/paas/v4/"
)

# Ollama client for local open-source models
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [4]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [5]:
# Get glm-5 to answer, with streaming

system_prompt = "You are a helpful technical assistant that explains code clearly and concisely."

stream = client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ],
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)

This line of code creates a **generator** that yields a list of **unique author names** from a collection of books, filtering out any books that do not have an author.

Here is the step-by-step breakdown:

### 1. The Set Comprehension
The code inside the curly braces `{...}` is a **set comprehension**:
```python
{book.get("author") for book in books if book.get("author")}
```
*   **`for book in books`**: Iterates through each dictionary (book) in the `books` list.
*   **`if book.get("author")`**: Filters the results. It keeps only the books where the "author" key exists and has a value that isn't empty (or `None`). This prevents `None` from appearing in your results.
*   **`{...}`**: By using curly braces instead of square brackets, Python creates a **Set**. This automatically removes duplicate author names. If "Orwell" appears five times in the book list, the set will only contain one "Orwell".

### 2. The `yield from`
The `yield from` keyword is used to delegate iteration.
*   Normally, `yield` yields a single value.
*   `yield from` takes an iterable (like the set created above) and yields every item from that sequence one by one to the caller.

### Summary of Execution
1.  It processes the list of books.
2.  It extracts the authors.
3.  It removes duplicates and `None` values.
4.  It passes those unique authors back to the caller one at a time.

### Example
If `books` is:
```python
books = [
    {"title": "1984", "author": "Orwell"},
    {"title": "Animal Farm", "author": "Orwell"},
    {"title": "Unknown", "author": None},
    {"title": "No Author Key"} # .get() returns None
]
```

The set comprehension creates: `{'Orwell'}`.
The line yields: `"Orwell"`.

### Why write it this way?
1.  **Memory Efficiency**: Because it uses `yield`, it creates a generator. If you had millions of books, this approach processes them one by one without loading a massive list of authors into memory all at once.
2.  **Data Cleaning**: The combination of the set and the `if` condition ensures the output is a clean stream of unique, valid author names.

In [8]:
# Get qwen-2.5-coder to answer (running locally via Ollama)

response = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
)

display(Markdown(response.choices[0].message.content))

```<|im_start|>
<|im_start|> This Python code uses the `yield from` statement to iterate over a set comprehension that extracts unique author names from a list of book dictionaries. Let's break down the code step-by-step:

1. `{book.get("author") for book in books if book.get("author")}`
   - This is a set comprehension.
   - It iterates over each `book` in the `books` list.
   - For each `book`, it attempts to retrieve the value associated with the key `"author"`.
   - The expression `book.get("author")` returns the author's name if it exists, and `None` otherwise.
   - The comprehension includes only those values that are not `None` (i.e., valid author names).

2. `yield from {book.get("author") for book in books if book.get("author")}`
   - This uses the `yield from` statement to delegate control to the set comprehension.
   - When an item is produced by the set comprehension, it is yielded back to the caller of `yield from`.

Summary:
- The code yields each unique author name found in the `books` list, skipping any entries where an author name is not present or is invalid (`None`).